# Phase 2: gradient boosting vs sklearn

From-scratch gradient boosting validated against sklearn on the BACE dataset (w/ Morgan fingerprints). One generic boosting core fits a sequence of shallow regression trees to the negative gradient of a loss and sums their scaled outputs; the task-specific part is a pluggable `Loss`.

Regression targets continuous pIC50 with squared-error loss (`loss=SQUARED_ERROR`); classification targets the binary active/inactive label with log loss (`loss=LOG_LOSS`). Each model is fit alongside its sklearn equivalent on one shared split per objective, so any difference is the algorithm, not the data.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "core").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error

from core.data import get_dataset
from core.Boost.boost import Boost, SQUARED_ERROR, LOG_LOSS

## Data

Load the cached BACE features once. Regression uses the continuous pIC50 target; classification uses the binary Class label on a stratified split. Each objective's models share one split.

In [2]:
X, y_class, y_reg = get_dataset()
y_c = y_class.astype(int)

print("X:", X.shape)
print("active / inactive:", np.bincount(y_c))
print("pIC50 range:", round(float(y_reg.min()), 2), "to", round(float(y_reg.max()), 2))

X: (1513, 2048)
active / inactive: [822 691]
pIC50 range: 2.54 to 10.52


## Regression - pIC50

Continuous target, squared-error loss. The booster starts at the mean and fits 100 depth-3 trees to the residuals, each added at a 0.1 learning rate. Scored with R^2 and RMSE against sklearn's `GradientBoostingRegressor` on the same split and the same hyperparameters.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=0,
)

ours = Boost(loss=SQUARED_ERROR, n_estimators=100, learning_rate=0.1, max_depth=3).fit(X_train, y_train)
skl = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_train, y_train)
ours_pred, skl_pred = ours.predict(X_test), skl.predict(X_test)

rb_ours_r2, rb_ours_rmse = r2_score(y_test, ours_pred), np.sqrt(mean_squared_error(y_test, ours_pred))
rb_skl_r2, rb_skl_rmse = r2_score(y_test, skl_pred), np.sqrt(mean_squared_error(y_test, skl_pred))
print(f"ours    R2 {rb_ours_r2:.4f}   RMSE {rb_ours_rmse:.4f}")
print(f"sklearn R2 {rb_skl_r2:.4f}   RMSE {rb_skl_rmse:.4f}")

ours    R2 0.6230   RMSE 0.8195
sklearn R2 0.6238   RMSE 0.8185


In [4]:
pd.DataFrame(
    {"ours R2": [rb_ours_r2],
     "sklearn R2": [rb_skl_r2],
     "ours RMSE": [rb_ours_rmse],
     "sklearn RMSE": [rb_skl_rmse]},
    index=["gradient boosting"],
).round(4)

,ours R2,sklearn R2,ours RMSE,sklearn RMSE
gradient boosting,0.623,0.6238,0.8195,0.8185


## Classification - active / inactive

Binary label, log loss. The booster accumulates a score in log-odds space (starting at the log-odds of the base rate), fits 100 depth-3 trees to the `y - sigmoid(F)` pseudo-residuals, and squashes the final score through a sigmoid to a probability. Predictions are thresholded at 0.5 for the accuracy comparison against sklearn's `GradientBoostingClassifier`.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_c, test_size=0.2, random_state=0, stratify=y_c,
)

ours = Boost(loss=LOG_LOSS, n_estimators=100, learning_rate=0.1, max_depth=3).fit(X_train, y_train)
skl = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_train, y_train)

ours_pred = (ours.predict(X_test) >= 0.5).astype(int)   # probabilities -> labels
skl_pred = skl.predict(X_test)

cb_ours = accuracy_score(y_test, ours_pred)
cb_skl = accuracy_score(y_test, skl_pred)
cb_agree = (ours_pred == skl_pred).mean()
print(f"ours {cb_ours:.4f}   sklearn {cb_skl:.4f}   agreement {cb_agree:.4f}")

ours 0.8152   sklearn 0.8482   agreement 0.9274


In [6]:
pd.DataFrame(
    {"ours": [cb_ours],
     "sklearn": [cb_skl],
     "agreement": [cb_agree]},
    index=["gradient boosting"],
).round(4)

,ours,sklearn,agreement
gradient boosting,0.8152,0.8482,0.9274


## Analysis

**Validation**
Regression almost exactly matches sklearn: R^2 ~0.62 vs ~0.62, RMSE ~0.82 for both. Close numbers validate the implementation. Classification lands close again but with a visible gap: accuracy ~0.82 vs ~0.85, with the two models agreeing on ~93% of test predictions. Reason for gap in classification is explained below. Both sit above a trivial baseline with no prediction besides mean (R^2 = 0 / ~0.54 accuracy), so the trees do carry the prediction.

**One core, two objectives**

The boosting function is identical for both classification and regression, the parameter `loss` swaps the calculations and output in the function. 
    - Regression: initial prediction is at the mean, residual = y - F, output = F (running predictions)
    - Classification: start at the log-odds of the base rate (the rate itself is near .5, so the starting score is near 0), residual = y - sigmoid(F), output = sigmoid(F)

The base tree is always a regression tree (variance + mean), even when doing classification - it only ever predicts signed continuous corrections (the residuals), which get summed into a score in log-odds space. Turning that final score into a 0 - 1 probability and then a label by proximity to 0 or 1 happens in the loss's output step, after all the trees, not inside any tree. 

**Why classification diverges more than regression**
The regression gap is just split tie-breaking similar to phase 1. The classification gap on the other hand is a real algorithmic difference: sklearn refines each leaf with a Newton step (the optimal log-loss leaf value), while mine uses the plain mean of the residuals. Adding more trees would hypothetically narrow the gap, since it is a per-tree difference rather than a ceiling; though that is not tested in this writeup.

**Takeaways**
- "Fit the residuals" is the squared-error special case of "fit the negative gradient of the loss."
- The running score F is the model's prediction, refined each round; predict just replays the stored trees from the start value.
- For classification, F lives in log-odds space, an unbounded score, and the sigmoid converts it to a probability only at the output that is made into a label.
- Shallow trees plus a small learning rate are what keep boosting from overfitting; no single tree can memorize the data in one shot.